In [ ]:
import os
import pandas as pd
from importlib import reload
import numpy as np
from tqdm import *
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
from model import params

In [ ]:
# 总目录
root_path = rf''
# 模型存储路径
model_path = rf'{root_path}/model_test'
# 打分存储路径
model_res_path = rf'{root_path}/model_res'
for path in [root_path, model_path, model_res_path]:
    os.makedirs(path, exist_ok=True)

In [ ]:
# 拼接各个fold的结果，使用zscore集成
def concat_model(basic_name):
    market = 'ALL'
    model_res = []
    for valid_period in ['validperiod1', 'validperiod2', 'validperiod3', 'validperiod4']:
        data_path = rf"{model_path}/{basic_name}/{market}-{valid_period}"
        date_list = list(set([x[:8] for x in os.listdir(data_path) if x > "2022"]))
        date_list.sort()
        all_res = []
        for date in date_list:
            date_res = pd.read_pickle(f'{data_path}/{date}.pkl').T
            date_res.index = [date] * len(date_res)
            all_res.append(date_res)
        all_res = pd.concat(all_res, axis=0).sort_index()
        all_res.index.name = "date"
        model_res.append(all_res.apply(lambda x: (x - x.mean()) / x.std(), axis=1))
    model_res = sum(model_res)
    model_res.reset_index().to_feather(rf"{model_res_path}/{market}_zscore_{basic_name}.fea")
    return model_res

In [ ]:
# 获取打分的ic均值和每期按流动性加权的label均值
def analysis_model(score, ret_data, liquid_data, money):
    start = '20230101'
    end = '20240630'
    model_score = score.copy()
    label_ret = []
    ic = []
    for date in model_score.loc[start:end].index:
        code_rank = model_score.loc[date].sort_values(ascending=False)
        ret = ret_data.loc[date].reindex(code_rank.index).fillna(0) * 100
        liquid = liquid_data.loc[date].reindex(code_rank.index).fillna(0)
        total_hold = 0
        total_earned = 0
        for num,code in enumerate(code_rank.index):
            if num >= 500:
                break
            if (money - total_hold) < 1:
                break
            hold_money = min(money - total_hold, liquid[code])
            total_hold += hold_money
            total_earned += ret[code] * hold_money
        total_ret = total_earned / money
        label_ret.append(total_ret)
        ic.append(code_rank.corr(ret))
        
    ic = pd.Series(ic, dtype='float').mean()
    label_ret = pd.Series(label_ret).mean()
    return dict(ic=ic, label_ret=label_ret)

In [ ]:
# 获取某个模型的集成打分
model_name = 'nn--fac20240819--label1--dropout0.2'
score = concat_model(model_name)

In [ ]:
# 查看打分的ic均值和每期按流动性加权的label均值
# ic和label均值越高越好
result = analysis_model(score, params.ret_data, params.liquid_data, 1.5e9)
result